In [0]:
dim_atleta = 'workspace.project_data_football_silver.dim_atleta'
dim_campeonato = 'workspace.project_data_football_silver.dim_campeonato'
dim_clube = 'workspace.project_data_football_silver.dim_clube'
dim_estadio = 'workspace.project_data_football_silver.dim_estadio'
dim_posicao = 'workspace.project_data_football_silver.dim_posicao'
dim_rodada = 'workspace.project_data_football_silver.dim_rodada'
fato_partida = 'workspace.project_data_football_silver.fato_partida'
fato_pontuacao = 'workspace.project_data_football_silver.fato_pontuacao'

In [0]:
spark.table(fato_partida).createOrReplaceTempView("tmp_silver_partida")
spark.table(dim_clube).createOrReplaceTempView("tmp_silver_clube")
spark.table(dim_campeonato).createOrReplaceTempView("tmp_silver_campeonato")

In [0]:
df_gold_classificacao_clubes = spark.sql("""

WITH brasileirao AS (

  -- Filtra apenas o Brasileirão antes de qualquer agregação
  SELECT 
    f.*,
    c.temporada
  FROM tmp_silver_partida f
  JOIN tmp_silver_campeonato c
    ON f.campeonato_id = c.campeonato_id
  WHERE c.nome = 'Brasileirão Série A'

),

base AS (

  -- Mandante
  SELECT
    temporada,
    clube_casa_id AS clube_id,
    gols_casa AS gols_pro,
    gols_visitante AS gols_contra,
    CASE
      WHEN gols_casa > gols_visitante THEN 3
      WHEN gols_casa = gols_visitante THEN 1
      ELSE 0
    END AS pontos,
    CASE WHEN gols_casa > gols_visitante THEN 1 ELSE 0 END AS vitoria,
    CASE WHEN gols_casa = gols_visitante THEN 1 ELSE 0 END AS empate
  FROM brasileirao

  UNION ALL

  -- Visitante
  SELECT
    temporada,
    clube_visitante_id AS clube_id,
    gols_visitante AS gols_pro,
    gols_casa AS gols_contra,
    CASE
      WHEN gols_visitante > gols_casa THEN 3
      WHEN gols_visitante = gols_casa THEN 1
      ELSE 0
    END AS pontos,
    CASE WHEN gols_visitante > gols_casa THEN 1 ELSE 0 END AS vitoria,
    CASE WHEN gols_visitante = gols_casa THEN 1 ELSE 0 END AS empate
  FROM brasileirao
),

classificacao AS (

  -- Consolidação por clube e temporada
  SELECT
    temporada,
    clube_id,
    COUNT(*) AS jogos,
    SUM(vitoria) AS vitorias,
    SUM(empate) AS empates,
    COUNT(*) - SUM(vitoria) - SUM(empate) AS derrotas,
    SUM(gols_pro) AS gols_pro,
    SUM(gols_contra) AS gols_contra,
    SUM(gols_pro - gols_contra) AS saldo_gols,
    SUM(pontos) AS pontos
  FROM base
  GROUP BY temporada, clube_id

),

ranking AS (

  -- Ranking por temporada
  SELECT
    *,
    RANK() OVER (
      PARTITION BY temporada
      ORDER BY
        pontos DESC,
        vitorias DESC,
        saldo_gols DESC,
        gols_pro DESC
    ) AS posicao
  FROM classificacao

)

SELECT
  r.temporada,
  r.posicao,

  r.clube_id,
  cl.nome_completo,

  r.jogos,
  r.vitorias,
  r.empates,
  r.derrotas,
  r.gols_pro,
  r.gols_contra,
  r.saldo_gols,
  r.pontos,

  ROUND((r.pontos * 100.0) / (r.jogos * 3), 2) AS aproveitamento_pct

FROM ranking r
LEFT JOIN tmp_silver_clube cl
  ON r.clube_id = cl.clube_id

ORDER BY
  r.temporada,
  r.posicao

""")

# Salvar como Delta
df_gold_classificacao_clubes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.project_data_football_gold.classificacao_brasileirao")

In [0]:
%sql
select * from workspace.project_data_football_gold.classificacao_brasileirao